<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/Notebooks/ClassicalRandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Classical Random Forest

In [ ]:
import numpy
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr

In [ ]:
def separateDFBySubtype(df,baseName):
    numericalCols = []
    categoricalCols = []
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numericalCols.append(str(col))
        else:
            categoricalCols.append(str(col))
    numericalDF = df[numericalCols]
    categoricalDF = df[categoricalCols]
    pickle.dump(numericalCols, open(f"../Data/Interim/{baseName}NumericalCols.pkl", 'wb'))
    pickle.dump(categoricalCols, open(f"../Data/Interim/{baseName}CategoricalCols.pkl", 'wb'))
    return numericalDF,categoricalDF

In [ ]:
def removeUnimportantCategoricalColumns(categoricalDF,y,combinedName,datasetType = 'train'):
    if datasetType == 'test':
        significantColumns = pickle.load(open(f"../Data/Interim/{combinedName}SignificantCategoricalCols.pkl", 'rb'))
    else:
        le = ce.OrdinalEncoder(return_df=True)
        leDF = le.fit_transform(categoricalDF)
        pValues = chi2(leDF, y)[1]
        pValueDF = pd.DataFrame({"feature":list(categoricalDF.columns),"pValue":pValues},columns=["feature","pValue"],index=None)
        lowPDF = pValueDF[pValueDF["pValue"] < 0.05]
        significantColumns = list(lowPDF['feature'])
        pickle.dump(significantColumns, open(f"../Data/Interim/{combinedName}SignificantCategoricalCols.pkl", 'wb'))
    return categoricalDF[significantColumns]

In [ ]:
def removeUnimportantNumericalColumns(numericalDF,y,combinedName,datasetType = 'train'):
    if datasetType == 'test':
        significantColumns = pickle.load(open(f"../Data/Interim/{combinedName}SignificantNumericalCols.pkl", 'rb'))
    else:
        numericalCols = list(numericalDF.columns)
        significantColumns = []
        allNumericalColumns = numericalDF.columns
        for col in allNumericalColumns:
            x = numericalDF[[col]].values.ravel()
            p = spearmanr(x,y)[1]
            if p < 0.05:
                significantColumns.append(str(col))
        pickle.dump(significantColumns, open(f"../Data/Interim/{combinedName}SignificantNumericalCols.pkl", 'wb'))
    return numericalDF[significantColumns]

In [ ]:
def processTestData(baseName):
    combinedName = baseName + balanceType
    df = pd.read_csv(f'../Data/Interim/{combinedName}Test.csv')
    yArray = df[['y']].values.ravel()
    df = df.drop('y',axis = 1)
    numericalDF = removeUnimportantNumericalColumns(df,yArray,combinedName,"test")
    categoricalDF = removeUnimportantCategoricalColumns(df,yArray,combinedName,"test")
    scaledDF = scaleTestDF(numericalDF,combinedName)
    encodedDF = encodeTestDF(categoricalDF,combinedName)
    finalDF = pd.concat([scaledDF,encodedDF],axis=1)
    finalDF['y'] = yArray.reshape(-1,1)
    finalDF.to_csv(f"../Data/Processed/{combinedName}Test.csv",index=False)

In [ ]:
def processTrainData(baseName):
  combinedName = baseName + balanceType
  df = pd.read_csv(f'../Data/Interim/{combinedName}Train.csv')
  yArray = df[['y']].values.ravel()
  df = df.drop('y',axis = 1)
  numericalDF,categoricalDF = separateDFBySubtype(df,baseName)
  numericalDF = removeUnimportantNumericalColumns(numericalDF,yArray,combinedName)
  categoricalDF = removeUnimportantCategoricalColumns(categoricalDF,yArray,combinedName)
  scaledDF = scaleDF(numericalDF,combinedName)
  encodedDF = encodeDF(categoricalDF,combinedName)
  finalDF = pd.concat([scaledDF,encodedDF],axis=1)
  finalDF['y'] = yArray.reshape(-1,1)
  finalDF.to_csv(f"../Data/Processed/{combinedName}Train.csv",index=False)


In [ ]:
def transformData(data):
  dateTimeColumns = ['First Contact','Last Contact', 'First Call', 'Signed up for a demo',
                     'Filled in customer survey','Did sign up to the platform','Account Manager assigned','Subscribed']
  for col in dateTimeColumns:
    data[col] = pd.to_datetime(data[col])
    data[col + 'Year'] = data[col].dt.year
    data[col + 'Month'] = data[col].dt.month
    data[col + 'Day'] = data[col].dt.day
    data[col] = data[col].astype('int64')
  return data

In [ ]:
def main():
  data = loadData()
  dateTransformedData = transformDate(data)
  np.random.seed(51)
  baseName = "SalesReinforcer"
  if exists(f"../Data/Interim/{baseName}Train.csv") == False:
      splitData(baseName)
  processTrainData(baseName)
  processTestData(baseName)


In [ ]:
if __name__ == "__main__":
    main()

Original datetime: 2023-01-01 12:00:00+00:00
Converted to int64 (nanoseconds since epoch): 1672574400000000000
Converted to seconds since epoch: 1672574400.0
